# Publica??o ?nica dos pesos da Fase 3.7

Execute no Colab pelo navegador. Este notebook usa Drive apenas uma vez para publicar o checkpoint no Release p?blico. A demo normal n?o usa Drive.


In [ ]:
# 1. Monte o Drive apenas para a publicacao inicial.
from google.colab import drive
from pathlib import Path
import hashlib
drive.mount("/content/drive")
CHECKPOINT=Path("/content/drive/MyDrive/TCC_Final/modelos/fase3_7_cfsmall_expansao_v1/phase3_final.pt")
EXPECTED_SHA256="cf3c9305dda7b01d05cef1520437470208b6cb3a8d45a37b4c5bdecd38215062"
if not CHECKPOINT.is_file(): raise FileNotFoundError(CHECKPOINT)
with CHECKPOINT.open("rb") as stream: actual=hashlib.file_digest(stream,"sha256").hexdigest()
if actual!=EXPECTED_SHA256: raise ValueError("Hash do checkpoint divergente")
print("Checkpoint verificado:",CHECKPOINT.name,CHECKPOINT.stat().st_size,actual)


In [ ]:
# 2. Publique uma unica vez; token solicitado em memoria, nunca impresso.
import getpass,json,urllib.request,urllib.error,urllib.parse
REPO="vicemnovo/TCC_Demo_Imagens_Fase3_7"
TAG="modelo-fase3-7-v1"
ASSET="fase3_7_cfsmall_expansao_v1_phase3_final.pt"
token=getpass.getpass("Token GitHub com permissao Contents: write: ").strip()
if not token: raise ValueError("Token ausente")
def api(url,method="GET",payload=None):
    req=urllib.request.Request(url,data=payload,method=method,headers={
        "Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","User-Agent":"TCC-Demo-Fase3-7"})
    try:
        with urllib.request.urlopen(req,timeout=120) as response: return json.loads(response.read())
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"GitHub HTTP {error.code} em {method} {urllib.parse.urlsplit(url).path}") from None
try:
    base=f"https://api.github.com/repos/{REPO}"
    try: release=api(f"{base}/releases/tags/{TAG}")
    except RuntimeError as error:
        if "HTTP 404" not in str(error): raise
        payload=json.dumps({"tag_name":TAG,"name":"Modelo Fase 3.7",
            "body":"Checkpoint final congelado. SHA-256: "+EXPECTED_SHA256,"draft":False}).encode()
        release=api(f"{base}/releases","POST",payload)
    if any(asset["name"]==ASSET for asset in release["assets"]):
        raise RuntimeError("Asset ja existe; nenhum arquivo foi substituido")
    upload_url=release["upload_url"].split("{",1)[0]+"?name="+urllib.parse.quote(ASSET)
    request=urllib.request.Request(upload_url,data=CHECKPOINT.read_bytes(),method="POST",headers={
        "Authorization":f"Bearer {token}","Accept":"application/vnd.github+json",
        "Content-Type":"application/octet-stream","User-Agent":"TCC-Demo-Fase3-7"})
    try:
        with urllib.request.urlopen(request,timeout=300) as response: asset=json.loads(response.read())
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"Upload falhou (HTTP {error.code}); confira o Release antes de repetir") from None
    if asset.get("name")!=ASSET or asset.get("size")!=CHECKPOINT.stat().st_size:
        raise RuntimeError("Asset retornou tamanho ou nome inesperado")
    print("Asset publicado:",asset["name"],asset["size"])
finally:
    del token


In [ ]:
# 3. Baixe publicamente para confirmar integridade.
import hashlib,urllib.request
public_url=f"https://github.com/{REPO}/releases/download/{TAG}/{ASSET}"
test_path=Path("/content/fase3_7_release_verificacao.pt")
with urllib.request.urlopen(urllib.request.Request(public_url,headers={"User-Agent":"TCC-Demo-Fase3-7"}),timeout=120) as response,test_path.open("wb") as output:
    while True:
        block=response.read(1024*1024)
        if not block: break
        output.write(block)
with test_path.open("rb") as stream: digest=hashlib.file_digest(stream,"sha256").hexdigest()
if digest!=EXPECTED_SHA256: raise RuntimeError("Asset publico com hash divergente")
print("Release publico validado:",public_url,digest)
